# Optimizing Public Transport Schedules Using Predictive Modeling

In [1]:
%%writefile app.py

#importing the packages
import streamlit as st
import numpy as np
import pandas as pd

# Function to load and preprocess the dataset
def load_data(file_path):
    data = pd.read_csv(file_path)

    # Preprocessing the data
    data["No. of buses of that type"] = pd.to_numeric(data["No. of buses of that type"], errors='coerce')
    data = data.dropna()
    data = pd.get_dummies(data, columns=["Type Of Bus (Ac / Non Ac)"], drop_first=True)
    
    # Select features and target
    X = data[["No. of bus terminals", "No. of bus stands", "No. of bus stops"]].values
    y = data["No. of buses of that type"].values

    # Normalize features
    X = (X - np.mean(X, axis=0)) / np.std(X, axis=0)

    # Add bias term
    X = np.c_[np.ones(X.shape[0]), X]
    return X, y

# Gradient descent implementation
def gradient_descent(X, y, alpha=0.01, epochs=1000):
    m, n = X.shape
    theta = np.zeros(n)
    for _ in range(epochs):
        gradients = (1 / m) * (X.T @ (X @ theta - y))
        theta -= alpha * gradients
    return theta

# Prediction function
def predict(X, theta):
    return X @ theta

# Main Streamlit app
def main():
    st.title("Public Transport Schedule Optimization")
    st.write("Predict the number of buses needed based on public transport data.")

    # File upload
    uploaded_file = 'C:/Users/Shuhaib/Downloads/shuaib/DATA SET/public-transport-accessibility-smart-cities.csv'#path of the file
    if uploaded_file is not None:
        X, y = load_data(uploaded_file)
        
        # Split dataset
        train_size = int(0.8 * len(X))
        X_train, X_test = X[:train_size], X[train_size:]
        y_train, y_test = y[:train_size], y[train_size:]

        # Train the model
        theta = gradient_descent(X_train, y_train, alpha=0.01, epochs=1000)

        # Evaluate the model
        y_pred = predict(X_test, theta)
        mse = np.mean((y_pred - y_test) ** 2)
        st.write(f"Model Training Complete. Mean Squared Error on Test Set: {mse:.2f}")
        st.write(f"Model Coefficients: {theta}")

        st.subheader("Make Predictions")
        # User input for prediction
        terminal = st.number_input("Enter the number of bus terminals:", min_value=0, value=10, step=1)
        stands = st.number_input("Enter the number of bus stands:", min_value=0, value=25, step=1)
        stops = st.number_input("Enter the number of bus stops:", min_value=0, value=100, step=1)

        if st.button("Predict"):
            # Normalize user input
            user_input = np.array([1, terminal, stands, stops])
            user_input[1:] = (user_input[1:] - np.mean(X[:, 1:], axis=0)) / np.std(X[:, 1:], axis=0)
            prediction = round(predict(user_input, theta))
            st.success(f"Predicted number of buses needed: {prediction:.2f}")

if __name__ == "__main__":
    main()


Overwriting app.py


In [ ]:
!streamlit run app.py